In [26]:
import pandas as pd
import glob
import os
import re
import sys
from omegaconf import OmegaConf

# Adjust the path to point to external/AlphaPEM
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))
from src.sampling.sampler import get_polarisation_curve_samples, build_fixed_parameters

data = pd.read_pickle('../sampling_test/all_samples.pkl')
data = data[~data['Ucell'].isna()]
parameter_ranges = OmegaConf.load('../param_config.yaml')

In [42]:
import time
import numpy as np
import matplotlib.pyplot as plt
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.multioutput import MultiOutputRegressor
from sklearn.gaussian_process.kernels import RBF

X = np.array(data[parameter_ranges])[:100]
Y = np.array(data['Ucell'].tolist())[:100]

def train_GP(X, Y, test_size=0.2, random_state=42):
    # Split data
    X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=test_size, random_state=random_state)

    # Scale input features
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    # Define kernel
    kernel = RBF(length_scale=1.0)

    # Initialize GP
    gp = GaussianProcessRegressor(n_restarts_optimizer=10, kernel=kernel, normalize_y=True)
    multi_gp = MultiOutputRegressor(gp, n_jobs=-1)

    # Measure fitting time
    start_time = time.time()
    multi_gp.fit(X_train_scaled, Y_train)
    elapsed_time = time.time() - start_time

    print(f"Fitting time: {elapsed_time:.2f} seconds")

    # Predict
    Y_pred = []
    Y_std = []
    for estimator in multi_gp.estimators_:
        mu, std = estimator.predict(X_test_scaled, return_std=True)
        Y_pred.append(mu)
        Y_std.append(std)

    Y_pred = np.array(Y_pred).T
    Y_std = np.array(Y_std).T

    # Evaluation
    mse_per_point = np.mean((Y_pred - Y_test) ** 2, axis=0)
    avg_mse = np.mean(mse_per_point)
    print("Average MSE across all time points:", avg_mse)

    return multi_gp, Y_test, Y_pred, Y_std, elapsed_time

# Use test_size=0.2 → 80% training
multi_gp, Y_test, Y_pred, Y_std, elapsed_time = train_GP(X, Y, test_size=0.2, random_state=42)


Fitting time: 0.39 seconds
Average MSE across all time points: 8.641430710957147e-05


In [43]:
from sklearn.metrics import r2_score

# Y_test and Y_pred are assumed to be NumPy arrays or pandas Series
r2 = r2_score(Y_test, Y_pred)
print(f"R² score: {r2:.4f}")

R² score: 0.9540
